[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tigi-prof-tigi/tgmaster-afe-python/blob/main/notebooks/Cours_10_Backtest/notebook_backtest.ipynb)

**Comment exécuter ce notebook** : clique sur le badge ci-dessus pour l'ouvrir dans Google Colab (gratuit, dans le navigateur, compte Google requis), puis exécute les cellules une par une (Maj+Entrée). Aucune installation.


In [ ]:
# === Setup (fonctionne en local ET dans Google Colab) ===
import sys, os
IN_COLAB = 'google.colab' in sys.modules
SRC, DATA = 'src', 'data/brvm_prices_sample.csv'
if IN_COLAB:
    import urllib.request
    BASE = 'https://raw.githubusercontent.com/tigi-prof-tigi/tgmaster-afe-python/main/notebooks/Cours_10_Backtest'
    for f in ['src/data_loader.py','src/strategy.py','src/backtest.py','src/metrics.py',
              'src/__init__.py','data/brvm_prices_sample.csv']:
        os.makedirs(os.path.dirname(f), exist_ok=True)
        if not os.path.exists(f):
            urllib.request.urlretrieve(f'{BASE}/{f}', f)
sys.path.append(SRC)


# Backtest momentum sur actions BRVM
**VERSION ETUDIANT - a completer.**

TgMaster University - Python pour la finance quantitative

Notebook gabarit. Completez les cellules marquees `# A COMPLETER`. Regle d'or anti look-ahead : le signal du jour J ne doit jamais utiliser le prix de J.


## 1. Chargement et nettoyage des prix
On utilise les modules de `src/`. Données fictives d'exemple dans `data/` (format Date, Ticker, Cloture).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_loader import charger_cours_brvm

prix = charger_cours_brvm(DATA)
prix.head()


## 2. Rendements quotidiens


In [ ]:
returns = prix.pct_change().dropna(how='all')
returns.head()


## 3. Signal momentum top 2 (anti look-ahead)
Performance glissante sur `FENETRE` jours, on garde les `N_TITRES` meilleurs, signal decale d'un jour.


In [ ]:
from strategy import signal_momentum

FENETRE = ... # A COMPLETER : fenetre du momentum (jours de bourse, ~3 mois)
N_TITRES = ... # A COMPLETER : nombre de titres a conserver
poids = signal_momentum(prix, fenetre=FENETRE, n_titres=N_TITRES)
poids.tail()


## 4. Backtest avec frais + référence buy-and-hold


In [ ]:
from backtest import backtester

equity, rend_strat = backtester(prix, poids, frais=0.005)
equity.tail()


## 5. Metriques : Sharpe et max drawdown


In [ ]:
from metrics import sharpe_annualise, max_drawdown

sharpe = sharpe_annualise(rend_strat)
mdd = max_drawdown(equity['Strategie'])
print(f"Performance stratégie    : {equity['Strategie'].iloc[-1]-1:.1%}")
print(f"Performance buy-and-hold  : {equity['BuyAndHold'].iloc[-1]-1:.1%}")
print(f'Ratio de Sharpe annualise : {sharpe:.2f}')
print(f'Maximum drawdown          : {mdd:.1%}')


## 6. Graphe : stratégie vs buy-and-hold


In [ ]:
equity.plot(figsize=(10, 5), title='Momentum BRVM vs Buy-and-Hold',
            color=['#1C2B5E', '#2645C8'])
plt.ylabel('Valeur (base 1)'); plt.tight_layout(); plt.show()


## 7. A vous
- A COMPLETER : tester une autre fenetre / un autre nombre de titres et commenter.
- A COMPLETER : exporter le graphe pour le rapport (`report/`).
- Rediger le rapport avec `Trame_Rapport_Resultats.docx`.
